## 1. Load dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("sales_data_sample.csv",encoding="latin1")
print("Original shape:", df.shape)
print(df.dtypes)

Original shape: (2823, 25)
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object


## 2. Clean and validate data types

In [2]:
df["PRODUCTLINE"] = df["PRODUCTLINE"].astype(str).str.strip().str.title()

df["SALES"] = (
    df["SALES"].astype(str)
    .str.replace(r"[^0-9.\\-]", "", regex=True)
)
df["SALES"] = pd.to_numeric(df["SALES"], errors="coerce")

print("\nData types after cleaning:")
print(df[["PRODUCTLINE", "SALES"]].dtypes)
print("Missing SALES values:", df["SALES"].isna().sum())


Data types after cleaning:
PRODUCTLINE     object
SALES          float64
dtype: object
Missing SALES values: 0


## 3. Detect revenue outliers using IQR

In [3]:
q1 = df["SALES"].quantile(0.25)
q3 = df["SALES"].quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

df["revenue_outlier"] = (
    (df["SALES"] < lower_limit) |
    (df["SALES"] > upper_limit)
)

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Revenue outliers:", int(df["revenue_outlier"].sum()))

Lower limit: -1253.4249999999993
Upper limit: 7964.855
Revenue outliers: 81


## 4. Automated console summary

In [4]:
print("\n--- SALES INTEGRITY REPORT ---")
print("Total records:", len(df))
print("Missing SALES values:", df["SALES"].isna().sum())
print("Revenue outliers:", int(df["revenue_outlier"].sum()))
print("Unique product lines:", df["PRODUCTLINE"].nunique())
print("Total sales:", df["SALES"].sum())
print("Average sales:", df["SALES"].mean())


--- SALES INTEGRITY REPORT ---
Total records: 2823
Missing SALES values: 0
Revenue outliers: 81
Unique product lines: 7
Total sales: 10032628.85
Average sales: 3553.889071909316


## 5. Product-line revenue report

In [5]:
product_summary = (
    df.groupby("PRODUCTLINE")["SALES"]
    .sum()
    .sort_values(ascending=False)
)

print("\n--- PRODUCT-LINE REVENUE SUMMARY ---")
print(product_summary)

print("\n--- HIGH REVENUE PRODUCTS ---")
for product_line, total_sales in product_summary.items():
    if total_sales > 100000:
        print(product_line, "-> High Revenue Product ->", round(total_sales, 2))


--- PRODUCT-LINE REVENUE SUMMARY ---
PRODUCTLINE
Classic Cars        3919615.66
Vintage Cars        1903150.84
Motorcycles         1166388.34
Trucks And Buses    1127789.84
Planes               975003.57
Ships                714437.13
Trains               226243.47
Name: SALES, dtype: float64

--- HIGH REVENUE PRODUCTS ---
Classic Cars -> High Revenue Product -> 3919615.66
Vintage Cars -> High Revenue Product -> 1903150.84
Motorcycles -> High Revenue Product -> 1166388.34
Trucks And Buses -> High Revenue Product -> 1127789.84
Planes -> High Revenue Product -> 975003.57
Ships -> High Revenue Product -> 714437.13
Trains -> High Revenue Product -> 226243.47


## 6. Documented cleaning logic

In [6]:
# Cleaning logic:
# 1. Strip whitespace and title-case PRODUCTLINE.
# 2. Remove non-numeric characters from SALES.
# 3. Convert SALES to numeric values.
# 4. Invalid SALES values become NaN.
# 5. Use the IQR method to identify revenue outliers.
# 6. Group SALES by PRODUCTLINE for the reporting summary.

## 7. Final summary

In [7]:
print("\n========== FINAL CONSOLE SUMMARY ==========")
print("Records analyzed:", len(df))
print("Unique product lines:", df["PRODUCTLINE"].nunique())
print("Total revenue:", round(df["SALES"].sum(), 2))
print("Average sale:", round(df["SALES"].mean(), 2))
print("Revenue outliers:", int(df["revenue_outlier"].sum()))
print("\nTop 3 product lines by revenue:")
print(product_summary.head(3))


========== FINAL CONSOLE SUMMARY ==========
Records analyzed: 2823
Unique product lines: 7
Total revenue: 10032628.85
Average sale: 3553.89
Revenue outliers: 81

Top 3 product lines by revenue:
PRODUCTLINE
Classic Cars    3919615.66
Vintage Cars    1903150.84
Motorcycles     1166388.34
Name: SALES, dtype: float64
